In [1]:
# Cell 1 : Install dependencies
!pip install requests beautifulsoup4 pandas rapidfuzz tqdm lxml -q
print("All libraries installed!")

All libraries installed!


In [2]:
import pandas as pd
import re
from datetime import datetime
from rapidfuzz import fuzz
from tqdm import tqdm

# ── Master schema ──────────────────────────────────────────
def create_empty_master():
    return pd.DataFrame(columns=[
        "id", "source", "name", "brand", "year", "gender",
        "top_notes", "middle_notes", "base_notes", "all_notes",
        "accords", "rating_avg", "rating_count", "longevity_avg",
        "sillage_avg", "price_tier", "season_winter", "season_spring",
        "season_summer", "season_autumn", "description",
        "parent_perfume", "flanker_type", "flanker_group", "is_flanker",
        "dupe_of", "image_url", "scraped_at"
    ])

# ── Text cleaner ───────────────────────────────────────────
def clean_text(text):
    if not text:
        return None
    cleaned = re.sub(r'\s+', ' ', str(text)).strip()
    return None if cleaned.lower() in ['nan', 'none', ''] else cleaned

# ── Price tier: Layer 1 — known brands ────────────────────
def build_brand_tier_dictionary(brands_df):
    brand_tiers = {}

    definite_luxury = {
        "creed", "xerjoff", "roja dove", "roja parfums", "clive christian",
        "maison francis kurkdjian", "mfk", "frederic malle", "bond no 9",
        "kilian", "by kilian", "amouage", "stefano ricci", "armani prive",
        "dior privee", "chanel les exclusifs", "tom ford private blend",
        "guerlain l'art et la matiere", "henry jacques", "baccarat rouge"
    }
    definite_budget = {
        "armaf", "lattafa", "rasasi", "al haramain", "afnan", "al rehab",
        "surrati", "rihanah", "fragrance world", "bharara", "ard al zaafaran",
        "maison alhambra", "pendora", "paris corner", "sterling parfums",
        "zimaya", "fa paris", "zara", "primark", "dunhill"
    }

    for b in definite_luxury:
        brand_tiers[b] = "luxury"
    for b in definite_budget:
        brand_tiers[b] = "budget"

    if brands_df is not None and len(brands_df) > 0:
        print(f"Reading {len(brands_df):,} brands from brands.csv...")
        for _, row in brands_df.iterrows():
            brand_name = str(
                row.get('name', '') or row.get('brand', '')
            ).lower().strip()
            if not brand_name or brand_name in brand_tiers:
                continue
            if 'price_tier' in row and pd.notna(row['price_tier']):
                brand_tiers[brand_name] = row['price_tier']
            elif 'category' in row and pd.notna(row['category']):
                cat = str(row['category']).lower()
                if any(x in cat for x in ['luxury', 'prestige', 'niche']):
                    brand_tiers[brand_name] = 'luxury'
                elif any(x in cat for x in ['premium', 'designer']):
                    brand_tiers[brand_name] = 'premium'
                elif any(x in cat for x in ['budget', 'mass', 'drugstore']):
                    brand_tiers[brand_name] = 'budget'

    print(f"Brand tier dictionary: {len(brand_tiers):,} brands mapped")
    return brand_tiers

# ── Price tier: Layer 2 — community votes ─────────────────
def build_community_price_map(perfumes_df):
    price_map = {}
    col = 'price_value_avg'
    if col not in perfumes_df.columns:
        print("No price_value_avg column found — skipping community price map")
        return price_map

    for _, row in perfumes_df.iterrows():
        name  = str(row.get('name',  '')).lower().strip()
        brand = str(row.get('brand', '')).lower().strip()
        key   = f"{name}_{brand}"
        price_avg  = row.get(col)
        vote_count = row.get('vote_count', 0)
        if pd.isna(price_avg) or pd.isna(vote_count) or vote_count < 50:
            continue
        if   price_avg <= 1.8: price_map[key] = 'luxury'
        elif price_avg <= 2.5: price_map[key] = 'premium'
        elif price_avg <= 3.5: price_map[key] = 'mid'
        else:                  price_map[key] = 'budget'

    print(f"Community price map: {len(price_map):,} perfumes mapped by voter consensus")
    return price_map

# ── Price tier: Layer 3 — keyword inference ───────────────
def keyword_infer_tier(brand_name):
    brand_lower = str(brand_name).lower()
    luxury_signals = [
        "maison", "atelier", "laboratoire", "parfums de", "privee", "prive",
        "exclusive", "prestige", "royal", "imperial", "house of",
        "editions de", "l'artisan", "artisan", "haute", "couture"
    ]
    budget_signals = ["inspired by", "impression of", "type", "generic"]
    if any(s in brand_lower for s in luxury_signals): return "luxury"
    if any(s in brand_lower for s in budget_signals): return "budget"
    return "mid"

# ── Price tier: master function (all 4 layers) ────────────
def infer_price_tier(perfume_name, brand_name,
                     brand_tier_dict, community_price_map, vote_count=0):
    brand_lower  = str(brand_name).lower().strip()
    perfume_key  = f"{str(perfume_name).lower().strip()}_{brand_lower}"

    # Layer 1: known brand dict
    if brand_lower in brand_tier_dict:
        tier = brand_tier_dict[brand_lower]
        # Layer 2 override: only if community strongly disagrees
        if perfume_key in community_price_map and vote_count >= 100:
            community_tier = community_price_map[perfume_key]
            conflict = (tier == 'luxury' and community_tier == 'budget') or \
                       (tier == 'budget' and community_tier == 'luxury')
            if conflict:
                return community_tier
        return tier

    # Layer 2: community votes alone
    if perfume_key in community_price_map and vote_count >= 100:
        return community_price_map[perfume_key]

    # Layer 3: keyword inference
    kw = keyword_infer_tier(brand_name)
    if kw != "mid":
        return kw

    # Layer 4: safe default
    return "mid"

# ── Flanker detection ──────────────────────────────────────
def extract_base_name(perfume_name):
    name = str(perfume_name).strip()
    concentration = r'\b(eau de parfum|eau de toilette|eau de cologne|' \
                    r'edp|edt|edc|extrait de parfum|pure parfum|parfum|eau)\b'
    name = re.sub(concentration, '', name, flags=re.IGNORECASE)
    flanker_words = {
        'intense', 'intenso', 'noir', 'elixir', 'absolue', 'absolute',
        'extreme', 'extrême', 'light', 'legere', 'légère', 'fresh', 'aqua',
        'sport', 'summer', 'eclat', 'éclat', 'gold', 'silver', 'rose',
        'rouge', 'blue', 'bleu', 'black', 'night', 'musk', 'oud', 'bloom',
        'neon', 'limited', 'edition', 'collector', 'ii', 'iii', '2', '3'
    }
    words = name.strip().split()
    while words and words[-1].lower().strip('.,') in flanker_words:
        words.pop()
    base = ' '.join(words).strip()
    return base if base else str(perfume_name)

def detect_flanker_type(perfume_name, base_name):
    name_lower = perfume_name.lower()
    if name_lower == base_name.lower():
        return "original"
    if any(x in name_lower for x in ['intense', 'elixir', 'absolue', 'oud', 'extreme']):
        return "flanker_stronger"
    if any(x in name_lower for x in ['light', 'legere', 'fresh', 'aqua', 'sport', 'summer']):
        return "flanker_lighter"
    if any(x in name_lower for x in ['noir', 'black', 'night']):
        return "flanker_darker"
    if any(x in name_lower for x in ['eclat', 'éclat', 'gold', 'rose', 'bloom', 'neon']):
        return "flanker_brighter"
    if any(x in name_lower for x in ['limited', 'edition', 'collector']):
        return "flanker_limited"
    return "flanker_variant"

def build_flanker_relationships(df):
    print(f"Detecting flanker relationships across {len(df):,} perfumes...")
    brand_groups = df.groupby('brand')['name'].apply(list).to_dict()

    parent_col, type_col, group_col, is_flanker_col = [], [], [], []

    for _, row in tqdm(df.iterrows(), total=len(df), desc="Flanker detection"):
        name      = str(row['name'])
        brand     = str(row['brand'])
        base_name = extract_base_name(name)
        siblings  = brand_groups.get(brand, [])
        found_parent = None

        for candidate in siblings:
            if candidate == name:
                continue
            sim      = fuzz.ratio(base_name.lower(), str(candidate).lower())
            cand_base_sim = fuzz.ratio(
                base_name.lower(), extract_base_name(candidate).lower()
            )
            if (sim >= 85 or cand_base_sim >= 90) and len(candidate) < len(name):
                found_parent = candidate
                break

        parent_col.append(found_parent)
        type_col.append(detect_flanker_type(name, base_name))
        group_col.append(f"{brand} {base_name}")
        is_flanker_col.append(found_parent is not None)

    df = df.copy()
    df['parent_perfume'] = parent_col
    df['flanker_type']   = type_col
    df['flanker_group']  = group_col
    df['is_flanker']     = is_flanker_col

    print(f"Flankers found: {sum(is_flanker_col):,} | "
          f"Families: {df['flanker_group'].nunique():,}")
    return df

print("All helper functions loaded!")

All helper functions loaded!


In [ ]:
print("Loading primary dataset...")
df_main = pd.read_csv('perfumes.csv', on_bad_lines='skip')
print(f"perfumes.csv:    {len(df_main):,} perfumes")
print(f"Columns: {df_main.columns.tolist()}\n")

print("Loading supplement ...")
df_supplement = pd.read_csv('fra_perfumes.csv', encoding='latin-1', on_bad_lines='skip')
print(f"fra_perfumes.csv: {len(df_supplement):,} perfumes")
print(f"Columns: {df_supplement.columns.tolist()}\n")

print("Loading FragDB reference tables...")
df_notes     = pd.read_csv('notes.csv',     sep='|', encoding='utf-8')
df_accords   = pd.read_csv('accords.csv',   sep='|', encoding='utf-8')
df_brands    = pd.read_csv('brands.csv',    sep='|', encoding='utf-8')
df_perfumers = pd.read_csv('perfumers.csv', encoding='latin-1', on_bad_lines='skip')

print(f"notes.csv:     {len(df_notes):,} rows    | Columns: {df_notes.columns.tolist()}")
print(f"accords.csv:   {len(df_accords):,} rows  | Columns: {df_accords.columns.tolist()}")
print(f"brands.csv:    {len(df_brands):,} rows   | Columns: {df_brands.columns.tolist()}")
print(f"perfumers.csv: {len(df_perfumers):,} rows | Columns: {df_perfumers.columns.tolist()}")
print("\nAll files loaded!")

<>:2: SyntaxWarning: invalid escape sequence '\p'
<>:2: SyntaxWarning: invalid escape sequence '\p'
C:\Users\Hp\AppData\Local\Temp\ipykernel_13568\1840894547.py:2: SyntaxWarning: invalid escape sequence '\p'
  df_main = pd.read_csv('data\perfumes.csv', on_bad_lines='skip')
C:\Users\Hp\AppData\Local\Temp\ipykernel_13568\1840894547.py:2: SyntaxWarning: invalid escape sequence '\p'
  df_main = pd.read_csv('data\perfumes.csv', on_bad_lines='skip')


Loading primary dataset...


FileNotFoundError: [Errno 2] No such file or directory: 'data\\perfumes.csv'

In [ ]:
# Build both lookup maps before mapping any perfumes
brand_tier_dict    = build_brand_tier_dictionary(df_brands)
community_price_map = build_community_price_map(df_main)
print("\nPrice tier lookups ready!")

In [ ]:
def map_fragrantica_to_master(df, brand_tier_dict, community_price_map):
    rows = []
    print(f"Mapping {len(df):,} perfumes from Archive 9...")

    for _, row in tqdm(df.iterrows(), total=len(df), desc="Mapping"):
        top   = clean_text(str(row.get('notes_top',    '') or ''))
        mid   = clean_text(str(row.get('notes_middle', '') or ''))
        base  = clean_text(str(row.get('notes_base',   '') or ''))
        flat  = clean_text(str(row.get('notes_flat',   '') or ''))

        all_notes = ' | '.join(
            n for n in [top, mid, base, flat] if n
        )

        rows.append({
            "id":            f"fra_{row.get('id', '')}",
            "source":        "fragrantica",
            "name":          clean_text(row.get('name')),
            "brand":         clean_text(row.get('brand')),
            "year":          row.get('year')   if pd.notna(row.get('year'))   else None,
            "gender":        clean_text(row.get('gender')),
            "top_notes":     top,
            "middle_notes":  mid,
            "base_notes":    base,
            "all_notes":     all_notes or None,
            "accords":       clean_text(str(row.get('accords', '') or '')),
            "rating_avg":    row.get('rating_avg'),
            "rating_count":  row.get('vote_count'),
            "longevity_avg": row.get('longevity_avg'),
            "sillage_avg":   row.get('sillage_avg'),
            "price_tier":    infer_price_tier(
                                 row.get('name', ''),
                                 row.get('brand', ''),
                                 brand_tier_dict,
                                 community_price_map,
                                 vote_count=row.get('vote_count', 0) or 0
                             ),
            "season_winter": row.get('winter'),
            "season_spring": row.get('spring'),
            "season_summer": row.get('summer'),
            "season_autumn": row.get('autumn'),
            "description":   clean_text(row.get('description')),
            "parent_perfume": None,
            "flanker_type":   None,
            "flanker_group":  None,
            "is_flanker":     False,
            "dupe_of":        None,
            "image_url":      f"https://fimgs.net/mdimg/perfume/375x500.{row.get('id')}.jpg"
                              if row.get('id') else None,
            "scraped_at":     row.get('scraped_at')
        })

    result = pd.DataFrame(rows)
    print(f"Mapped {len(result):,} perfumes successfully!")
    return result

fra_master = map_fragrantica_to_master(df_main, brand_tier_dict, community_price_map)
fra_master.head(3)

In [ ]:
# Cell 6 : Map fra_perfumes.csv to master schema

def extract_brand_from_url(url):
    """
    Fragrantica URLs look like:
    https://www.fragrantica.com/perfume/Giorgio-Armani/Si-Passione-12345.html
    Brand is always the segment after /perfume/
    """
    try:
        parts = str(url).split('/perfume/')
        if len(parts) > 1:
            brand_slug = parts[1].split('/')[0]
            # Convert slug to readable name: "Giorgio-Armani" → "Giorgio Armani"
            return brand_slug.replace('-', ' ').strip()
        return None
    except:
        return None

def extract_notes_from_description(description):
    """
    fra_perfumes.csv has no separate note columns.
    The description often contains note info like:
    "Top notes are Bergamot and Lemon; middle notes are Rose..."
    We extract what we can.
    """
    if not description:
        return None, None, None

    desc = str(description)
    top, mid, base = None, None, None

    top_match  = re.search(
        r'[Tt]op notes?\s(?:are\s)?(.+?)(?:\.|;|middle|heart|base|$)',
        desc
    )
    mid_match  = re.search(
        r'(?:middle|heart) notes?\s(?:are\s)?(.+?)(?:\.|;|base|$)',
        desc, re.IGNORECASE
    )
    base_match = re.search(
        r'[Bb]ase notes?\s(?:are\s)?(.+?)(?:\.|;|$)',
        desc
    )

    if top_match:  top  = clean_text(top_match.group(1))
    if mid_match:  mid  = clean_text(mid_match.group(1))
    if base_match: base = clean_text(base_match.group(1))

    return top, mid, base

def clean_accords(accords_raw):
    """
    Accords come as a string like: ['citrus', 'musky', 'woody']
    Convert to clean comma separated string.
    """
    if not accords_raw:
        return None
    cleaned = re.sub(r"[\[\]']", '', str(accords_raw))
    cleaned = ', '.join([a.strip() for a in cleaned.split(',') if a.strip()])
    return cleaned if cleaned else None

def map_fra_supplement_to_master(df, brand_tier_dict, community_price_map):
    rows = []
    skipped = 0
    print(f"Mapping {len(df):,} perfumes from Archive 7...")

    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Mapping supplement"):

        # Name — column is capitalised 'Name' in this file
        name = clean_text(row.get('Name'))
        if not name:
            skipped += 1
            continue

        # Brand — extracted from URL since there's no brand column
        brand = clean_text(extract_brand_from_url(row.get('url', '')))
        if not brand:
            skipped += 1
            continue

        # Gender
        gender_raw = str(row.get('Gender', '')).lower()
        if 'women' in gender_raw and 'men' in gender_raw:
            gender = 'unisex'
        elif 'women' in gender_raw:
            gender = 'female'
        elif 'men' in gender_raw:
            gender = 'male'
        else:
            gender = clean_text(row.get('Gender'))

        # Accords
        accords = clean_accords(row.get('Main Accords'))

        # Notes — extracted from description
        top, mid, base = extract_notes_from_description(row.get('Description'))
        all_notes = ' | '.join(n for n in [top, mid, base] if n) or None

        # Rating
        try:
            rating_avg = float(row.get('Rating Value', 0) or 0)
            rating_avg = rating_avg if rating_avg > 0 else None
        except:
            rating_avg = None

        try:
            rating_count = int(row.get('Rating Count', 0) or 0)
            rating_count = rating_count if rating_count > 0 else None
        except:
            rating_count = None

        rows.append({
            "id":             f"fra7_{idx}",
            "source":         "fragrantica_supplement",
            "name":           name,
            "brand":          brand,
            "year":           None,
            "gender":         gender,
            "top_notes":      top,
            "middle_notes":   mid,
            "base_notes":     base,
            "all_notes":      all_notes,
            "accords":        accords,
            "rating_avg":     rating_avg,
            "rating_count":   rating_count,
            "longevity_avg":  None,
            "sillage_avg":    None,
            "price_tier":     infer_price_tier(
                                  name, brand,
                                  brand_tier_dict, community_price_map,
                                  vote_count=rating_count or 0
                              ),
            "season_winter":  None,
            "season_spring":  None,
            "season_summer":  None,
            "season_autumn":  None,
            "description":    clean_text(row.get('Description')),
            "parent_perfume": None,
            "flanker_type":   None,
            "flanker_group":  None,
            "is_flanker":     False,
            "dupe_of":        None,
            "image_url":      None,
            "scraped_at":     None
        })

    result = pd.DataFrame(rows)
    print(f"\nMapped:  {len(result):,} perfumes")
    print(f"Skipped: {skipped:,} (missing name or brand)")

    # Quick sanity check
    print(f"\nSample entry:")
    if len(result) > 0:
        sample = result.iloc[0]
        print(f"  Name:    {sample['name']}")
        print(f"  Brand:   {sample['brand']}")
        print(f"  Accords: {sample['accords']}")
        print(f"  Gender:  {sample['gender']}")
        print(f"  Rating:  {sample['rating_avg']} ({sample['rating_count']} votes)")

    return result

fra_supplement = map_fra_supplement_to_master(
    df_supplement, brand_tier_dict, community_price_map
)

In [ ]:
# Cell 6b : Clean concatenated names in fra_supplement

def clean_supplement_name(name, brand):
    """
    fra_perfumes.csv has names like:
    "9am Afnanfor women"  → should be "9am"
    "Si PassioneGiorgio Armanifor women" → should be "Si Passione"

    The brand name and gender got fused into the perfume name.
    We strip them out.
    """
    if not name or not brand:
        return name

    cleaned = str(name)

    # Remove brand name from within the perfume name (case insensitive)
    cleaned = re.sub(re.escape(str(brand)), '', cleaned, flags=re.IGNORECASE)

    # Remove gender suffixes that got fused in
    gender_suffixes = [
        'for women', 'for men', 'for women and men',
        'for unisex', 'unisex', 'women', 'men'
    ]
    for suffix in gender_suffixes:
        cleaned = re.sub(
            rf'\s*{re.escape(suffix)}\s*$', '',
            cleaned, flags=re.IGNORECASE
        )

    cleaned = clean_text(cleaned)
    return cleaned if cleaned else name  # fallback to original if we stripped too much

print("Cleaning concatenated names in supplement dataset...")
before = fra_supplement['name'].iloc[0]

fra_supplement['name'] = fra_supplement.apply(
    lambda r: clean_supplement_name(r['name'], r['brand']), axis=1
)

after = fra_supplement['name'].iloc[0]
print(f"Before: '{before}'")
print(f"After:  '{after}'")

# Spot check a few more
print("\nSpot check (10 random names after cleaning):")
print(fra_supplement[['name', 'brand']].sample(10).to_string(index=False))

In [ ]:
print("Running flanker detection on primary dataset...")
fra_master = build_flanker_relationships(fra_master)

print("\nRunning flanker detection on supplement dataset...")
fra_supplement = build_flanker_relationships(fra_supplement)

print("\nDone! Both datasets have flanker relationships mapped.")

In [ ]:
# Cell 8 :  Merge Into Master Dataset

def merge_fragrantica_sources(primary_df, supplement_df):
    print(f"Primary:    {len(primary_df):,} perfumes")
    print(f"Supplement: {len(supplement_df):,} perfumes")

    # Reset indexes first to avoid duplicate index errors
    primary_df    = primary_df.reset_index(drop=True).copy()
    supplement_df = supplement_df.reset_index(drop=True).copy()

    def make_key(name, brand):
        n = re.sub(
            r'\b(edp|edt|edc|eau de parfum|eau de toilette|eau de cologne)\b',
            '', str(name).lower().strip()
        ).strip()
        b = str(brand).lower().strip()
        return f"{n}|{b}"

    primary_df['_key']    = primary_df.apply(
        lambda r: make_key(r['name'], r['brand']), axis=1)
    supplement_df['_key'] = supplement_df.apply(
        lambda r: make_key(r['name'], r['brand']), axis=1)

    primary_keys       = set(primary_df['_key'])
    supplement_new     = supplement_df[~supplement_df['_key'].isin(primary_keys)].copy()
    supplement_overlap = supplement_df[supplement_df['_key'].isin(primary_keys)].copy()

    print(f"Overlap (already in primary): {len(supplement_overlap):,}")
    print(f"Genuinely new in supplement:  {len(supplement_new):,}")

    # Fill empty fields in primary from supplement where possible
    fill_cols = [
        'accords', 'top_notes', 'middle_notes',
        'base_notes', 'all_notes', 'description', 'image_url'
    ]

    # Build lookup from supplement — drop duplicate keys, keep first
    supplement_overlap_deduped = supplement_overlap.drop_duplicates(
        subset=['_key'], keep='first'
    ).set_index('_key')
    supp_map = supplement_overlap_deduped.to_dict('index')

    filled = 0
    for idx, row in primary_df.iterrows():
        key = row['_key']
        if key not in supp_map:
            continue
        for col in fill_cols:
            val   = primary_df.at[idx, col]
            empty = pd.isna(val) or str(val).strip() in ['', 'nan', 'None']
            if empty:
                new_val = supp_map[key].get(col)
                if new_val and str(new_val).strip() not in ['', 'nan', 'None']:
                    primary_df.at[idx, col] = new_val
                    filled += 1

    print(f"Filled {filled:,} empty fields from supplement")

    master = pd.concat([primary_df, supplement_new], ignore_index=True)
    master = master.drop(columns=['_key'])

    print(f"\nFinal master dataset: {len(master):,} perfumes")
    return master

master_df = merge_fragrantica_sources(fra_master, fra_supplement)

In [ ]:
print("=== MASTER DATASET HEALTH CHECK ===")
print(f"Total perfumes:        {len(master_df):,}")
print(f"With notes:            {master_df['all_notes'].notna().sum():,}")
print(f"With accords:          {master_df['accords'].notna().sum():,}")
print(f"With ratings:          {master_df['rating_avg'].notna().sum():,}")
print(f"With descriptions:     {master_df['description'].notna().sum():,}")
print(f"Flankers identified:   {master_df['is_flanker'].sum():,}")
print(f"Fragrance families:    {master_df['flanker_group'].nunique():,}")
print(f"\nPrice tier distribution:\n{master_df['price_tier'].value_counts()}")
print(f"\nGender distribution:\n{master_df['gender'].value_counts()}")
print(f"\nTop 10 brands:\n{master_df['brand'].value_counts().head(10)}")

In [ ]:
# Cell 9b : Compute derived columns from existing data

def enrich_master_dataset(df):
    print("Enriching master dataset with derived columns...")

    # 1. Note complexity — count unique notes per perfume
    def count_notes(all_notes):
        if pd.isna(all_notes) or not all_notes:
            return 0
        notes = re.split(r'[|,]', str(all_notes))
        return len([n.strip() for n in notes if n.strip()])

    df['note_complexity'] = df['all_notes'].apply(count_notes)

    # 2. Dominant note — first note listed is usually most prominent
    def get_dominant_note(top_notes, all_notes):
        source = top_notes if pd.notna(top_notes) and top_notes else all_notes
        if not source or pd.isna(source):
            return None
        first = re.split(r'[|,]', str(source))[0]
        return clean_text(first)

    df['dominant_note'] = df.apply(
        lambda r: get_dominant_note(r['top_notes'], r['all_notes']), axis=1
    )

    # 3. Olfactive family — derived from accords
    def infer_olfactive_family(accords, all_notes):
        source = str(accords or '') + ' ' + str(all_notes or '')
        source = source.lower()

        families = {
            'Floral':    ['floral', 'rose', 'jasmine', 'lily', 'violet',
                         'peony', 'iris', 'magnolia', 'tuberose'],
            'Oriental':  ['oriental', 'amber', 'vanilla', 'musk', 'oud',
                         'incense', 'spicy', 'warm spicy', 'balsamic'],
            'Woody':     ['woody', 'cedar', 'sandalwood', 'vetiver',
                         'patchouli', 'oakmoss', 'pine', 'birch'],
            'Fresh':     ['fresh', 'citrus', 'aquatic', 'green', 'marine',
                         'bergamot', 'lemon', 'lime', 'grapefruit'],
            'Gourmand':  ['gourmand', 'sweet', 'caramel', 'chocolate',
                         'coffee', 'praline', 'honey', 'almond'],
            'Fougere':   ['fougere', 'fougère', 'lavender', 'coumarin',
                         'aromatic', 'herbal'],
            'Chypre':    ['chypre', 'mossy', 'earthy', 'oakmoss',
                         'labdanum', 'cistus']
        }

        scores = {family: 0 for family in families}
        for family, keywords in families.items():
            for kw in keywords:
                if kw in source:
                    scores[family] += 1

        best = max(scores, key=scores.get)
        return best if scores[best] > 0 else 'Other'

    df['olfactive_family'] = df.apply(
        lambda r: infer_olfactive_family(r['accords'], r['all_notes']), axis=1
    )

    # 4. Best season — pick the season with the highest community vote
    def get_best_season(row):
        seasons = {
            'Winter': row.get('season_winter'),
            'Spring': row.get('season_spring'),
            'Summer': row.get('season_summer'),
            'Autumn': row.get('season_autumn')
        }
        valid = {k: v for k, v in seasons.items()
                if pd.notna(v) and v is not None}
        if not valid:
            return None
        return max(valid, key=valid.get)

    df['best_season'] = df.apply(get_best_season, axis=1)

    # 5. Data completeness score — useful for filtering low quality entries
    key_cols = [
        'name', 'brand', 'all_notes', 'accords',
        'rating_avg', 'description', 'gender', 'year'
    ]
    def completeness(row):
        filled = sum(
            1 for col in key_cols
            if pd.notna(row.get(col)) and
            str(row.get(col)).strip() not in ['', 'nan', 'None']
        )
        return round(filled / len(key_cols), 2)

    df['data_completeness'] = df.apply(completeness, axis=1)

    # 6. Confidence score — based on how many people voted
    def confidence(vote_count):
        if pd.isna(vote_count) or vote_count == 0:
            return 'low'
        elif vote_count < 50:
            return 'low'
        elif vote_count < 500:
            return 'medium'
        elif vote_count < 5000:
            return 'high'
        else:
            return 'very_high'

    df['confidence_score'] = df['rating_count'].apply(confidence)

    # 7. Limited edition flag — from name signals
    def is_limited(name, flanker_type):
        if pd.isna(name):
            return False
        name_lower = str(name).lower()
        signals = ['limited', 'edition', 'collector', 'anniversary',
                  'exclusive', 'special', 'holiday', 'summer 20',
                  'winter 20', 'spring 20']
        return any(s in name_lower for s in signals) or \
               flanker_type == 'flanker_limited'

    df['limited_edition'] = df.apply(
        lambda r: is_limited(r['name'], r['flanker_type']), axis=1
    )

    # 8. Occasion tags — inferred from accords + notes + description
    def infer_occasions(row):
        source = ' '.join(filter(None, [
            str(row.get('accords', '') or ''),
            str(row.get('all_notes', '') or ''),
            str(row.get('description', '') or '')
        ])).lower()

        tags = []
        occasion_map = {
            'office':   ['clean', 'fresh', 'light', 'subtle', 'professional',
                        'soft', 'aquatic', 'citrus', 'white floral'],
            'date':     ['seductive', 'sensual', 'warm', 'amber', 'vanilla',
                        'musk', 'intense', 'night', 'romantic', 'oriental'],
            'casual':   ['casual', 'everyday', 'fresh', 'citrus', 'green',
                        'light', 'easy', 'simple'],
            'wedding':  ['floral', 'rose', 'peony', 'white floral', 'elegant',
                        'bridal', 'delicate', 'powdery'],
            'sport':    ['sport', 'aquatic', 'fresh', 'energetic', 'marine',
                        'clean', 'ozonic'],
            'evening':  ['evening', 'night', 'intense', 'rich', 'oud',
                        'smoky', 'leather', 'dark', 'deep']
        }

        for occasion, keywords in occasion_map.items():
            if sum(1 for kw in keywords if kw in source) >= 2:
                tags.append(occasion)

        return ', '.join(tags) if tags else 'everyday'

    df['occasion_tags'] = df.apply(infer_occasions, axis=1)

    # 9. Days since release — useful for surfacing new perfumes
    current_year = 2026
    df['days_since_release'] = df['year'].apply(
        lambda y: int((current_year - int(y)) * 365)
        if pd.notna(y) and str(y).strip() not in ['', 'nan'] else None
    )

    print(f"Enrichment complete! Added 9 derived columns.")
    print(f"\nOlfactive family distribution:")
    print(df['olfactive_family'].value_counts())
    print(f"\nBest season distribution:")
    print(df['best_season'].value_counts())
    print(f"\nOccasion tags distribution:")
    print(df['occasion_tags'].value_counts().head(10))
    print(f"\nConfidence score distribution:")
    print(df['confidence_score'].value_counts())
    print(f"\nData completeness (avg): {df['data_completeness'].mean():.2%}")

    return df

master_df = enrich_master_dataset(master_df)

In [ ]:
# Cell 9c : Add confidence weight for recommender use

confidence_weights = {
    'very_high': 1.0,
    'high':      0.8,
    'medium':    0.5,
    'low':       0.2
}

master_df['confidence_weight'] = master_df['confidence_score'].map(
    confidence_weights
)

# Also add a popularity score combining rating + vote count
# This prevents obscure perfumes from outranking well-loved ones
def compute_popularity_score(rating_avg, rating_count, confidence_weight):
    if pd.isna(rating_avg) or pd.isna(rating_count):
        return 0.0
    # Bayesian-style score: blend actual rating with a prior of 3.5
    # The more votes, the more we trust the actual rating
    prior      = 3.5
    prior_weight = 50  # acts like 50 "neutral" votes
    bayesian_rating = (
        (prior * prior_weight) + (rating_avg * rating_count)
    ) / (prior_weight + rating_count)

    # Scale by confidence
    return round(bayesian_rating * confidence_weight, 4)

master_df['popularity_score'] = master_df.apply(
    lambda r: compute_popularity_score(
        r['rating_avg'],
        r['rating_count'],
        r['confidence_weight']
    ), axis=1
)

print("Confidence weights and popularity scores added!")
print(f"\nPopularity score stats:")
print(master_df['popularity_score'].describe().round(3))
print(f"\nTop 10 perfumes by popularity score:")
print(
    master_df[master_df['popularity_score'] > 0]
    .nlargest(10, 'popularity_score')
    [['name', 'brand', 'rating_avg', 'rating_count', 'popularity_score']]
    .to_string(index=False)
)

In [ ]:
master_df.to_csv('master_dataset.csv', index=False)
print(f"master_dataset.csv saved — {len(master_df):,} perfumes")

lean_cols = [
    'id', 'name', 'brand', 'year', 'gender',
    'top_notes', 'middle_notes', 'base_notes', 'all_notes',
    'accords', 'rating_avg', 'rating_count', 'longevity_avg',
    'sillage_avg', 'price_tier', 'season_spring', 'season_summer',
    'season_autumn', 'season_winter', 'description',
    'parent_perfume', 'flanker_type', 'flanker_group', 'is_flanker',
    'source', 'image_url'
]
lean_df = master_df[[c for c in lean_cols if c in master_df.columns]]
lean_df.to_csv('master_dataset_lean.csv', index=False)
print(f"master_dataset_lean.csv saved")

from google.colab import files
files.download('master_dataset.csv')
files.download('master_dataset_lean.csv')
print("\nSave both files into Beyond-Fragrancy/data/")